# SGBR E1 trên Kaggle: 10 run nặng trên NZEnergy (1 JVM, heap 26 GB)

Sgb(Srp) và Srp(Sgbt) trên NZEnergy cần nhiều bộ nhớ: ở heap 6 GB, cả 4 run thử ở shard 1 đều tràn bộ nhớ hoặc treo ở khoảng 40–50%. Bài báo dùng heap 48 GB (tr.34). Notebook này chạy **mỗi session đúng 1 run** với `-Xmx26g` (Kaggle cho 30 GiB). Quyết định này do người phụ trách duyệt ngày 2026-11-21.

| `HEAVY` | Phương pháp | Seed |
|---|---|---|
| 1–5 | Sgb(Srp) = `SGBT(SRP(FIRTDD))` | 1–5 |
| 6–10 | Srp(Sgbt) = `SRP(SGBT(FIRTDD))` | 1–5 |

**Mỗi người làm 4 bước:**
1. Kaggle → **New Notebook → File → Import Notebook**, chọn file này.
2. Settings: **Accelerator → None**, **Internet → On**.
3. Đổi `HEAVY` thành số được giao (1 đến 10). Mỗi số chỉ một người chạy.
4. **Save Version → Save & Run All (Commit)**. Khi xong, tải `E1_heavy<N>.zip` ở tab **Output**.

**Lưu ý:**
- Thời gian ước tính khoảng 6–9 giờ cho mỗi run. Ở tốc độ máy tác giả, các run này mất 6,1–6,4 giờ, và thời gian Sgb(Srp) dao động mạnh giữa các seed. Nếu run chưa xong ở mốc 11,6 giờ, notebook dừng, ghi tiến độ (%) rồi vẫn tạo file zip.
- Nếu hết bộ nhớ, run dừng ngay (cờ `-XX:+ExitOnOutOfMemoryError`) và được ghi là `oom`, thay vì treo hàng giờ.


In [ ]:
HEAVY = 1                # <-- 1..5 = Sgb(Srp) seed 1..5; 6..10 = Srp(Sgbt) seed 1..5
METHOD = ["SGBT(SRP(FIRTDD))", "SRP(SGBT(FIRTDD))"][(HEAVY - 1) // 5]
SEED = (HEAVY - 1) % 5 + 1
XMX = "default=26g"      # One JVM per session; Kaggle gives 30 GiB
SESSION_BUDGET_HOURS = 11.6
RESUME_ZIP = None
TAG = f"E1_heavy{HEAVY}"
assert 1 <= HEAVY <= 10, "Set HEAVY between 1 and 10"
print(f"Heavy task {HEAVY}: NZEnergy x {METHOD} x seed {SEED}")
RUN_ARGS = ["--exp", "E1", "--datasets", "NZEnergy", "--methods", METHOD, "--seeds", str(SEED),
            "--jobs", "1", "--xmx", XMX, "--exit-on-oom"]


In [ ]:
import base64, hashlib, sys, zlib
from pathlib import Path

WORK = Path("/kaggle/working")
SCRIPTS = WORK / "sgbr_scripts"
SCRIPTS.mkdir(parents=True, exist_ok=True)
EMBEDDED = {
 "kaggle_environment_smoke.py": "eNqdGu1y2zbyf2byDijbachYoiwnaXK6Uzu6xEncxIkndnK9sVUOREISY5JgAVK2k/rdbxcASZCi3PTUSU0Cuwtgv3fB778blVKMFnE2YtmG5DfFmmeP7t9zHOeYUVkKRih5G2flNcl4wRacXxLJpIx5RmgWkQ0T8fKGFGtakLiQ5NcXb0hIMyLKjIQ8Yj4Qun/v/r04zbkoCBWrnArJ6oGIFqyI02bgs+RZ/ZLSYl2/cFk/5gktllyk9YBoCMh1WcRJ81oucsFD2HI9VLA0X8ZJg1KKJIkXPhOCi+6gYH+UTBb37y0FT0kOG4JRYmBO1P7wv48np2cfDmfHZEqcdVHkcjIareJiXS78kKejrLyi2arMqGSXVNDR6at/f3AarOD5++PjozNEXv50sHy6ePzo6ZPxAds/2F8+eRo+XoTR/qNnzx6Hj58ePGYRfbIYA/bx7NPhu2D24ezo5ew5Irv37xH41RsQLOd+Sjcs82lOwzXzuViN1MDBCB+B17S4ydmIS4n/hiAclhWjp9bL8Kmf8xSW8+7fO/nvyVFwfHg2ezE7m9lHzW/yWBHHh1FI85uU0xHKEhDfzF69ensY/Of9hzdH714BGrLNdUaXdLVK2OiKi8s4Wzme5mTElqg9LrAtBQ0bkPAqmr7jGfMm+niFuDFP+AOwPGEFi4BuI2sfKTRABrChB//gYc3Cy+lLmkgGLzQvQNsDXhZ5WUzPRAmDBbuuH0FJYW76aL8h6+lHdh2yvCDu+9ND1KCBvY8zjXZ4nceCRR6hkig1s04gGCycka8Ou46LAI3GmRA88IA4ejfwLgvhKkTvVmNWWA2dFn7NFl8D4vDAgq0Juw2kLCIYJnukNQSrevBXxLnrGQq3jaSUAgRw2gVzwWL6ZXQFhtCxJx9eec4yRGqYO95XHBJM5jyTbNKWYM0oXDSQBS1KCSeooH09ctsSim3a/uuzsxMlojvF0KauMZF9tzuk/YkmJVPP3t+gawSsoLvyrZgbrgQv8yCJU3CtbsrSBRNyHedTjZzyMivibMlb5gEO9wOj4JljGS8SRjbj0eYAfDWoYwEn18SI8mbFmhGjqA8koWFR0kRTldpvI714SZqVSSzVzq3zWZO1ZSPRkWTJcqSP4HggdxoFaE+u19CtDtBHtp7roVpP9hE2GwK8r0ZkS3XujJE4s7bryzyJCxyXrmetHIAz4FkheJIA4MBgADkE1UiuM3EG5MBrkHCJBgsXsmhUSAPH66i02c15AzyvzqunlFNESCO3KTmf64EwL4M/Sl5QexCQuLgJwJnZoxCkMWDjkGQ1n1pcqRi6iykKYEAwbMobCRG0ww8yJMCRscUR8OngY4GZPC/M4g22QfPOJ4/mDQqoBGJhpoGbch2jPWAn+ulgi4HIOMhOWGengeC8MBYSYNBuomP1U0wWcIBy4Qrn4sI93x8+nX99dOvBeglNFxHF/CNcgzNdCzfOCle9no/nA/LM85AZLIk8r00WuaomarbWZ300eTLfCh5KPgewPXXy6bQ5qh3jalVCrXacW2QVYDEIXUqkhseWonnkR4AEJUHuabVwbu/SVwkJDYtce60fK/30L9kN6EO/9sKWetS4DVk7ETyhMectWeJPnR0p1pJrw7CkJuXHMhAMMsF4w4KCu43gv40wGVWEdlDprrwVkPoVULEXNR1ltY3gYNBFwUBmdt0IR715gx54lGGFEi6lNvoAgsiA1GM5pOE8wsFeEkb+SMWsptxJEGfB4qZgQKqLdrtblqg6EWQzIYSTG2WnircD8lD99XXaKPuEALJD266x+0WIwunDxt8C3P3l9tQGY7Dt8rdNMmEZTZWzU8LZQb+duPQvc14RQ1/tNqwY1au0olKVPPWTNemEySbuWLtf0QxX9cZ2YFfu36dR5B7YrmO8Y1M62EDRABla5H51UCImSamPCw7w4UO97m0PGaWlA6IVE/ikIf0VOCtb+YdjMoYgblzkDjI96B1DUI896PXy6L7Ni3X+PqqNKcH29h3P62W43td34K7xIKoUxiX0PsjPZF8NmRXhdYdomiBeMduiMjL4PTvQVtzhi+VIBn0TXZvXe+874Pcm9YTkEfLXHDJstGhSZooE1FpmfZXYU6iKV2xIk3iVwdTb9+9eBcez3/xevhnELcbpcY/8ixw8fPjTLnZZ6Y3NL4Nc50o9tZGxAVBiZ32zEHHk4G7AWt3KOlBqY60WGdotnFM0kwOVn9pe0tFGAhT1gz1VixXKMWAewKQxVLS1sAcE0ntaJsV0i6o5opaVFpTGts7ej24XDrASZnN1CteTSAPMzgwaqp0EvSmkR0tIS2iSQIr0uxom6N8u5MMJ/HP9Pe8H0COkM0Dg449vz47eHr07rFJMVe1lYcy2qcH65Pj1l28lVZdRenPojXTCgumPHsOczOELycSGRUG6/tIBorm7TDjFNLbZl+fZFdfnTQoBSfJShFYCvEzoSh2gzNxz5zPdUOW6fvttsnciQAFf4vzLOKOJGq90bW62DiULuEzVoYjisHAtPlhlovM7sOHiau9C7rmzECPiiS7NuHiOYfHPY3r9mtH8NP7C/vwoGYC+eu4h+36ZIgMvTvcgc7UyPNzUeVXoz9tMNXCet9NgFDoy0Gze1lHgJoQiNd1liVzzq1MzPdG5focnNiGjxn2EKt4+122bt1BrKDb3U7NEmFIwFipWm1ZVrLqOmKpWHUh/JlZlCm7tRM24EZMhhGlMo6dBEPEwCDwbFYNnQA2O6wyHhrEDgg20KVpWY5bazkCR8HAj2H+8hOobO6yx4BlS8FVvzLt7iYSHSqWgMsdtOVjAQ54kSgaDa5bkU+cTuF3soRLYjowjRt6opppzN2HTwR0qLzNcg7rL6iDKQv4OtjaWekN6ffLxiGCayEOFRXH/Kg8r1lDqGwJ/sUnUhaFWP1wN28/1Kh8YlYaigchWEIiw+5yxK4br0Iw8+4sVwAL51TCNYTfZaqj6ondz+yW2jiANgngXkeP3M/KZim9ao8wuM36VtVnXv5YhBzTQ5Riq6g/SlUq1mxYKptM47CtdmbRqaZxr914x34YEzu1m12YV1YACvS0zTJO1JAcox5xKSYw+KqZTIJ4Njaw3WgctieLPHFlHMm14ELSRG0E1ZbhhQDAVUECt4UBpZ9Ug2jpfexUk0Snzv50i/uBoSBXvHYBX4KUhC3B3kfC2cHcu9q8p2e9dqQdDG5TVKLhbWAxSjEi18ow4quuZhiXYODJmWudIpusOgum057eUSudE6NL88CqqcoSCc5UifMUH477xcaACz9zT1l4PqDqxOYfr6Jsm5c1rdw7x221CwNZo2Bm2yaWbbBtppWysu0CZfYlzDVwTqbrqBlAnK5JREa5d8aAaddxfJuML3/vFvYj2vAcDzQQTteZNrDUswuEgpZ+5MAWIoXM+BvZgFWLIKv42uqiS6uqOyj9jeNVExc2LqvByIR9fxtfgNFYLMYSYMoTDgFVPjUhVF7ou0yzlUVGpngAGnGDb3le79/wrAbqu08C2vj0IEzR/BUy+krxcJHGITrCAPxseRzrknkKVm63O50pzPAB80CFzqjt/wCQ/x8ieZK6D92DBr7NPs+D9G8f757fgfMBOZcqwsjGPUGLDEeKELpImZ8J21f9PD2qTY5VvKyq35PYiA3EzvEaBQ06dslgOnzm9zTy8PAHBBep2qZU0Kv21eD7Xd1FNRd0QSWiZheuaxpbray1ybt38zLHB1i2fOtQ6qVaojMGvN3fXvlT77Y7FjVNVXsdSfxhGn9I39fOUjMed/X5PTpW3GqIFqGqE0ERywq6ZCGPwb8rdLcosSiAOm+0IAqakWi05DGImEKcQp6X/N3hxt3BMrqyVpOUiWmm8rdOY0sMC4+E/5uf78L+H3vbID3bWbm+vcSiqVrU3rlyG4/Qn9Zb34XjGFmZXFO0Fu4qkQKoTW6hmiQ1eA3Rqpjo1MRuwZAuQfpljuurWBhvktcVO0UeaxdBHDsCxXAem7lWlcAviYO61CiyMoOq8VugCBbO98JQ8wyFLXt+opICHQwvw9zodUJPNupicei32Q+zR16K1kplolMihgCOpZBZCE3J8qDg+INXnADDx+nD2og4kdbfHrt9hcFf9vuayqPh2GS9M9LEimvM7eLczXkDGiOUmRjNnYJbpVN8oCE01iiVyV39f4eNbUEqwMLeKO2azptppd79NngJ1OS2Csgih4qs+/fDrB8jk3PoF//cFZOEDtAfJGMcPPmjhtgpIq6LC1o4umFD5uumLoz82sE3Nqb4iAczq0a8e2svoZCVomkgNfGumjZXwVQwbCMJc3fhy6WNfJsRivg1IlyrTvKkgsS0F0BKMMgogLlXz7r7XwuvIGTA7I62elL5Lm3Rulb1O3woB6u6RPdcyc4CCd3vaKEGAegGzrZ5xpSCg1wUqXd3XQmBfDcHUUjDWnsGR23bjYSu1BthdWfduTJMMT3an3zauyvIAWP1tsaT2EMiP+mULpO0nqmV7prYwTWwzH3TUQXcLTntxVB3LndtQGStQCiiZdjx0yhxExGgaoMdagysBmNp5dW6FHFUjByGYmwBDm7Q+/2h/jtS9UHLwy6AORutzIhuhJXR2nasOVFDvFD/lidHgO99PdYUWriExDWTKLyHMQA7L8Gyd7LYKk+jeW3FE1ZJW0WraBIFiATHFDST/WHnMtyOnlTRZTaq6ItWR3dyV+eklVuXm4sx8cAQEwZb5pXr1tlGtfB17SX5Uprl0K++LndQIqE0PPLJHnAusjrrpa9W4wETYtUi3m4L7qm1vyJ7v4uvcXCjpPhxgBAE2ioNAXWkHAZYIQeCYZEBQyOKITsYPgXGuqiAwnP8PKYxJiQ==",
 "e1_memory_feasibility.py": "eNrFO2tz2ziS31OV/4DgKhvSkWg7mUntqkY7q8ROJhPHdtmema1VdCxKhCTGFMkjKD/O0X/f7gZAgg/ZztxWXaqSUGR3o9EvdKOB/3q2u5b57jRKdkVyxbLbYpkmr58+4ZyfrRMWJCFbiUCuc8HSRLD5Oo7Z4T6bCfh/nYQiZwHbf8M+RG/Zr8FVwJYiyACSHUXJ+sYDKk+fPH0SrbI0L1iQL7Igl6J8MZNX5fMykMs4mpa/v8o0KX+ksnzMK3S5XBdRXP2MFklg/VxPszydCVnhFtGqwv7fKJtHMfye5+mKZUGBwzP98RR+Iuf07TJYLGLhg3yiPE1WIil8uUovhQH+NPrw4ejQ/+Pk7NPH4w899tvp+cXZ4eiz/+7k8+ePFz02W+TpOvPjaBUVEsk+fXJw+H7029GFfzC6GJ0fXrAh48f/OkxEvrjl1dejw9HZ8eEZfj3/8PbCGZ29PxML5/3Hs4uDA9cFyF9Hv4/8k9OLjyfH/u+js4+jt0eH5wB/x+nLxcnJkf58znuM/3rwybdQ6F39xebpk5OTz/7p6OLi8OwYKOXCm6WrDETlPH3C4E8+5Sfr4mT+WazS/PYwz9P823FQRFcCTAVfsSCO0xm8SRNvZx4AZvjtYinAhCLJokSu5/NoFoEYNTzvacLC+/jh+OTs8B2IBN65SlShmLNFVPjT20JIJxdZ2iN1uQODVqzzxNK3N1uK2aWfrotsXThjDsg4z/47+FcWOZFw4YVcptfwas7vGhrbDO5wgA2fWCwApghWNS7CoAikKHrsOs0vDTvrTEGC6OZc3GS7ZwcKTO7eaYSNF+TzOVfwQT5bouyGRIXtMp7m0SICU/bARDWQRvSRLQtSjWRTK/JbzQf+uY4A3Mb10kwkDr+ecpcFUs/JQqCxolzMChjFkmi+Tpw6FP75c6I1AgLx9to0ZRGC4oYKBIlCjMmHFiunH08PwaVQxcP3QSxFg4hb/YzmejKespFZGorGZElCWgUN4WB0GLQZBKVkIvSjBIzrMUL6DwiKDKFLWJbAkNs/I66GyLTY7EluFx55XxBJwX4P4rWgUODMOfg6y6IkEWHlDMQA8MnCVEiWpBD806QIIlxiGIS192ye5qxyEO526EmHbO9fUfYe45HWG2lLcdzBoMGEr0rBIOu6HypXSNf5TPS+21tKNdBiBKEyu0Ue0+lXx5BUeHpCc/Ts2HZSY33rJI6SS2cVSRklCz+9HF7ka+HWgpzNXRWazFtxFYUimQnHDo9htBASLVWvsJ5cBq9+fONowlHiIzp8J+tQL2fpOkGUPfWTRGgJJN8mENQhTAICfbIttiAz3jqDMYWDoA09X6Ehwcj4yQMSUea0rVPz3KEJxfjLIZumaewoYpjDoMHRL6AZ5IXEGTlT/py7DeoiBvoKMk6vRe64bAjU+D9wQN4xYiU/1JaW6VxbODBjobRcxfYU8gFQEWCSPlmeXsvSDbT677hSHh8YSS7FjXpyMJrAcgPRjhwX5gmWIAGS2NhU1iL2fVjQVyAWB3xwHi2shSwWQQ5JiJ8EKzRcIcIe+wp5Hf6bG4uKSoujaYKuFZ0x168lnzCwBANkASijgO/3iuVcxLgCQWQwSyHmnIbcNJhdgsCmkGeA15Fjk/RKYQF/yHmTuRxmnK58/PTnGCCilBHus2IJKd1iyX4sR0Xbr8Tmz+NgUcrTZkO/qrOAFgO4aGu2Chr2NgXhXapXAnz1kVMQNwE8GE5oGJoE5ibw11eMeZhw2yK0+UC++OifH97m/EF7DjJYeq5AUiCxLIbxVdrLRjeLKStShs9SQCEAEYC9S3PR399jsHTKpv5IhNZwZgYvMbFi/bsSZsPoecONaIwBPBtuUf54b/LAPCBTNeNpp1Tah+E8pmW7j9OZp5DsXoM9UIHUh4nAZGCN4pQ6IvmvV5gLGkbQm/w0w+QYGPFkFkeFCXEAOYa/XgSF1Y3D+/9c3fzw1wV3J1gA4K/9NwuT5wXyEt5a+cb8xZ0ZQ2AAC4o055MN60vmjGCZwxXzXC3GfZPN0vrnshcWFd6PmXOn575xWT9kmhym81CxaQbqgWmsggTvz+JASlwqdJ6DYaPHdmBWPeJ5UsUhYCD0C3FT1JaregKrydPaU8HrscXNTGQFOzknrbWxVPFpDUaREGK6PZ5ZcBrc1NceTTBKCrWmuKxcJSIZilm0CmJYKdAp2TGYQTWuqnB8mQQZpHqFQ5WgNIOb1xIrtk0VR6A2gjIKgoYCh6BBPlQLGboUwErVIfgxJ7lP6gkwVxx4q+CGI0UCbcSVDMJKD9SMlS0mL2huGg0/YeKqfxKM5BU6OZv5SlzC2qOKpD8zHHDpr2WwEBWVavAUHBizxjyNaxzIZpxEoCgxq3GZF0DQteTczFfqBgDVVcWgq9wUIaXjNgabRyIOZZmy2P5sqSHGxJMgKaF4RVmJejHen9hWtC2rAe7HGmFvghEBrbGkUFe6Ldthp/yak4AqfZ0LaRzB9hQsNDUFBJslBW/PryQQqeweveC+mXBDCmdikCv40jHGGETIIxHwrk6RoyVpMxl0Mo4AbqPi4RrGV/k5YFbi2gJaplHwaIFs6hsQhueW94cihtjnTAXYG+QHwRzoVLsXch0XDf8n7sEmFYZ3KW7B7thfFKb+aWcPSVREkKMMNcIY8SfjivuJZfKUzQwVqfsAFWMKoi16yk4UsTE+T1jfsKF+N+xLJ0c4KYeDIaJboz1eRnFsntUumXmz3dYMITNtSvD1OzU95VwVb38fbmOurkI1Z2vPJ7gSPmTK0RwzbBRFT8OUEbyAYBWCfGiJwrLCp/0tWL5VRlWsMjMFBetd51EhVKQhiHC9ynBHCcn2GK79STF85bKXjH9JQBICS2+oCId8Xcz7f21QywUkWabkqzhXaUa6cuytKbRwdPGx2T9CKKhzFjDKDr7zFnE6dfhS+iLP/SwKd+irO2mbJtFqbjV11IhL0EVrHwWUGCS3jrXR6EmBdbAqCWtxWRHoCotaaVXdpV/oQtaIQu+E2FWRA3mMb2cBpmLTryEeUzioeZmmrtZ3SkgDqqurFySDkgbJIRHXOJEh71Jkp3xw7lD7Uckgr7yDaFacQWSDmNYtCc0FoFSpO76CQNwIwq0J1JdmU/TP4zQoHKShixVg2coBWVVbTuqJmPNJqO3gHru4zYR+rPLqe4VppVjECKVY9ITKgNdiQdV4I8c6PTv5cHZ4fu6fHp69Ozy+qG9YgxE6P//0bPwl9Cau8yW82++93jg/D754X8KX7s/uc3SmM8A+vxhd/HbeQOb/7fy+gq/ffl/98sdnd/BFvnQQj12+5RgHvM9uI8EEU1uAI0sHvKbHZkGe30INCIWOLAMGNlYAlIFhMFq5VDGG9QM1UAgzSmbxGk2FaETg6H0tH7VzIlVnhWxuiav8bLlOLtlgiOiUKDtvfvzx9Ru3YVeroJiR9zbF5kG0DCNcOIlpiD1EsWlrJllWJkLU6rmHNsA99tNQA8PD/t5eh/cqsTANZ5khMQClTZ2Tcf/1q8GkZis1+VqLbgqhPoXlPVhlqEYdWFUq3WMQ1cAVY1ircXPDaMyK58Va1uoByrDnfBfjyO4doG92FRR3yUjhjZ31KAs16skl5ZywZA/s8mEHhPLqB9IJfOppWYFiKmMklQRx7KjBXL1SzdZ5LpLi/7dg0EzUk7P7EkeNYC/mf6Zy+O5xG7WEiVeygFy7EDp4kxK1HZiwqbSIGtN5kNnb4pAudWyzTUClJY1dNPnGzl01EX4NWvVncTq79KWAhDzE9NJYpAWnDLsMKn4mIF0FIQ7KkWxoMymKQs0FDwfQ320cyhLARKscWkpvIQqHU+Djduqsku37MCBI1jF031NnwVp5Jab+bSMYSN0vpfkD5JYiWmNaW5thBPVXcFvFYRUDMEZMRTzkh/vcOLqWN5gAdoWx3J/jg8Of/zJ4/nnwHFuj9GWxoveK0rhLeROziVwawJAZ8G06nNQxfGIQC+HrAPJUCPzo1Z9PRny7cc75nfkw8F7NN895M0IkGL70LjjSK6OM4e4+BU08ApfNYtYKFZpeFfvcnvrPbH2p1u1UR72OHZp1AvYUweRjwWsUa9NUb3cpZu7svKbZ4pEDbsbJcvRUexdrfKcVvJmwOxLuBjcZKZf5xk6N3NhdXQMb+MhtOrQmgysAJE5EC+5F3Q1eTFxEVNLUkFqY7oZbBj6P13JJbZ1euYtW1Yp0zII8zDE1hyk3VIu5WsTQJNN1gTnlVbWIUZqUg7SGr/cgo+9qGNanSx6B3XR4MQ/l0HHrZem44ZGqvKSafYtPlqtoXlTOBStyWsCqPDO2RN3ALI3ISjWwaUfBDPJ1prCr9FCv9ZaHTTGlbuTejtOqbNxGEw/zK1su2zFMBVOJbBsqfKWtBF+nHlT5xdXQHbl/j02hTBQ5vtqnYTSu7SU6kNe7zKdEtK5aoy3d1+ETSD6vw6GyGd0gpsSy3R8+vzg4+e2CDGmojMkYg3mw5uw2twdMJ0/lKEOuXYyrrhFWkEMzEDxb6Jg0lf3Fdi1CRlQrvpse0ewPUxqMntWRalLKh4HVV3z2GOUFwEAqPXz/g2Nx2cO3fxyf/DI6/tAR+1TGt6ULbHVoarslUM9t84UGbe3ZtYxSdTCvWb90rL8PDeAWRup+xBE4xBMx/Lv4prL4GkerPHbLgG0HrVdENS9qQm9ZYlS0xWDzQEpvK68mqAeHaJgx0d8Capyb9m/srZs7ruuBAbMcYGdHUduYnZytk/wOM6/66FsSnXp834JdC78or5ftJaSNqfKkWIjM2fN+tDN5Vf5D9T9Ngzz8aEyvuelTN0nrJ29sT8a37f0idIZuv2tHDvMH/Bh3E+vOrY4JeucfP3z6eHTUISE9nVOFdJSml+us2WGqRZdAyvYX/7sizl6Dj/JIUXn0pkJVFP0i9cVNVOA3xxqpVmHVUlwr8lQOUkH7/wHnJe35j3VbjG1dpXg3vXFnAYKpCInWy9fgRTdYb6viettSZdO0hnqUd6PAwVzRuW0ytovb6ZMZ0nYVUBSqbNih4WjepXcIvnvVrpe1GIP7rIAJ3JPzlVEP+4+m+lMHUdck7ditqFI83b24PyPsbckYdQ1n9hiU9gQem8X1KMXOeH2rWqd7VS8AexXJrdPu5amy02oi7Ll46qT9udZX2FO7y2VDEKohNcFGuePW82BsWTTaILw2GTCP1uQ6amHTe1Jj4jkeerAhQZfIK4BGySwKgRS1FTSBcr6Dkre2ISCrw0akoyzCkmtVPdubv12b5SbNbZwDaEioa9dhQgcfgHK5NW1F/UGr12Q8TXc3DaB1rKNtP7hbbuto/Fj5Te4fnywnnUpYDMsFilhoO9MDhIhndQzapqMk/mzIHrnB9MAoMNcUY1EhfHW2g3cfFOpANYihdV5iFUSJE+SLqyEV9JoCHd/P0SP1UX5vlC/WeCL+lL44oZCzPKLTLUPfD9OZ77s2qheEoR9oHIf3++aAaN8cEMVtl9tMDE91s+1/1lEuQvso5BZKqzTofw3y/xO+Pl33XXhGbT3c7QhAuMPG2f770fVJmw50ffj/fnQ8lWSmDC5TUdm/Hw+N0hqTjJQ/ICCdmtYlbCjQtjJX5iV3Deyu2De7S3MRyGgaxVFxS/VxefRyy3C6vumbXbZqkg8INJ0FGOiDGVkhWDosUH4BeoSXSxFnQ35Em20QzaJQtbRgHAmRgn2imx4PyMHkKX2TKnfK//Xe/VSMi/crF3/sDIGBr4DbX0RFHyv+qNxXByhJvWjCo/8QU5InV6fr8JXXyvixVbNX6xcQFaGOxHXMm63WsmBTwbJURnj/wz7Bh/k6jUP6KA/h1u/KYGsvjPJam7U+LF5CwkPiWjXs3elvSEdMIS3vMepHQ8anta76xvRUKddmihjShmW2b1uVdifQA7JpGOuDksmFh1epqIPm5Hy81/9b0J+P+u8nd29+2KD1IhMQknwVktx7xq4CVzksdhNzYCWH1c8cAE/n9FofN/58MsIjvdYxReOyKrrD4Pq3h7tb8ZUwiVH5HvgAg/RWl6hB9UOqvU1Mc2XROMFOWcSwQq+flaDjBJaQSjAihadfQLV0RMG82CqRd8s0hcQ2oBZrOStqDBUpLN+C1nTcM8bGmT4yXyqIxlBzcpsndOwkrawJgCCMTzW/3Y2pLoshkApKpQkqG1WNQnW7rIYsZ2mG55J488pdDcosnb6OAYPWxTO7eaNihr+g+kPDmzjQ+GLjmdVtwO44Hp0xWOYGRa1fpZcyDaJ/1iYmKFGmz3S4vJlN4/k/vEJYNon23+hybmfndQsad2sH2CAgpSF7m93qSIutjTJz9TFzbeI0Elsbs7m12znmlv3fx/XyGlXdg328JvzjoMrCs+NbR/3QAVQPiEaLjdf1ouJRW1vNQyh4hBmrQqzA4zQIpVNd/qMhK7vXKaO6+NA8y1672gGhroxrEFQx8pmwhgd9Z4Udqcp2AoI94jRT64YNBjNfX8tQGHTTi3JL176y0biqRmSeDZvB39xEaQzbPraug3p1x0qd96itBue/jPpIk7vtokCLRg9LxYFiyhbkFd510bedrpcRrF/ELn6oz4dA26eQulmnPtt1oNieQ2kU4jH7gqlMFQM3zuJK5NE80svXdB3FIfv14FPnXMpWCHDbcePFDmK9eryywpN198Xaw0muyiMdVW+1fowjlZ5eAbyoECtcv0AiAGMupXTe2920pwHuCOlr6FvXBvwrWG6wY6rKN0m1sNN9EfgvNifq3GiHrIg4yFYCdUVzy13auhbLQ/8l6n0NJjxLo9MDHTSoS2n6TlaTqX3vq7xRW7uEuy0a1HVr7SrZcy6rbbNJ17pEp6/t1Y2aaLfL81YS+dhTI8bb2x8fcpk5P7VujqndhDDCnqJ1pMyQZXdbRtlUx/lqbqT2aHH7xr677nQ5mg1A1qNwra3V7+lsd64qqs3dXbbUzyDe14To3jZpdiHMYUZ9rcQ+wNizHeO8fNTfzO3Ut0GoL6j2mDkRSYuHaF1T6WqhEhR2jPF/vAft0JO79f6o03Ub3L3/OumjV+do3iG22p5RrV9NdxL4GPLP06PDi8ODCWauwBU4Z4iHop6zb+xOE3zRkbm8mAx6myrM27VxdXbCtQ99QCZm6CkGX0w2kKya6Wy6MPXpk71HzE4l6ftqYwzAfboP5/sE5vu4Teb7RgbKR89vJYT8Q0i4HNpFQ839G9HQr6A=",
 "sgbr_c01_e1_integrated.py": "eNqdO2tz27aW3zvT/4ByJmsqFSnRsuTHXO3WiZ0mbR17bCe7t46WQ5GgxFgiWT78iK/2t+85BwAJUnSSXn+wRBA4L5w3IMMwLrLkLsqjJO6zO55F4SPz4oBlZcyKJWd5FC9W3IqTgs+T5Jb97i3gmd0n2W24Su5twzB+/OHHH6J1mmQF8/O76nvgFbyI1rwaWHr5chXNq+fPeRJXD2uvWFYPSV59TVdeESbZuhrIaoD5siyiVf0YLWJPeyznaZb4PK+BFXydhtGqhtCgr8xWQJ6d8b9KnhfV8JdIrgmzZM1SoBNmMfnygsiu5nLHXfN1kj26IffyaB6togLkmcOLas4tSdDl8V2UJfGax4Wbr5NbjtP8oYPQfvzh8vTN6eXp+9en7tn5sXv19nh3PGFTZhzO+Wi0Ox/uj8Z7vjPaHe1zLzj0Dg6CvZHjHYzHXuA5e8PhaHQ4ns/3h6ODCQ/9cTieeP7erjOBzXpzfO3+dnzpvj8+O0WQ68Szdoe7I3u4ZzvW1fvji6u359fWZy+z7qNiaQU85XHAYz/iuQ2jAOK3k9/dk3eXp6+vzy//iTA+B7eW49hDe/fw532YcHb88fR9c4qXev6SW2vvjsfWyD60HQchHX8E9k7/gHnvzt+778+viaaz09+PmQOT9lkk9DCN4pgHDKTBQB1uGe5SlPGceQVbgawL9pt35zEgwhASRBoVi+fAATw7zgcLKHUfJnvuKorLB3eZFHmaFK6k3d23Cy+zF18kjx8u/4DlobEsijQ/GgwWII9ybvvJeuAFSVpE5XoAOlVmUew41jyKvQyENMg4UgRfguQ+XiVeMNDk82L31f7gSVG3kZjqHR75B+HufDwKAme07wewf+O9cG8YhIeTMBjCXk6CcH/X39879IeTvckhBw2Yw86Oh6O5v++HlfQV8x2CR1JrTsX0Nq8ZTxObFtkCgp1kiwEN7A7wqxgVIwMdyUAgGTzVdGwqNMDo2NkFTOaPPzD4M+Z+yPdCPhqO/cPJrudzVNX9+Tj0973hwfxgPxh78+Bgvs8PDoe7+8Dr2BlODkOPzzkfO5P50JCQAh4cHE6cvb1xMAkOHe47hwfjAEGHB854tDs5ONgD6R563uhwd380ChxvxP3D8cQ/PNw/ACg9oTkBDxls8RrcoOlli7zP/PugT74iKYupszvsHQmUoH/lqgBmal9jg+OUrOFftX4qYPCHYnqdlbzP8iJAaNrKi3cXp/16KUzgWaZPuLo+Of9wXVMiPwH+kvu30zfeKucSQE98RKGkEbxaUWaxnwT8qEaReVHO2WUZI6TTLEsyMzSekOab4WzD+ENUgM09bYHYfIrVoGDjxhoPh8Oj2cboKcngXNaYVEs3iBbgYU30pX3mrRZJBoa1nhr50gMrMJR40f2Qh6WZPTsBMzaNbG700FfmRca9tc6NwCnDjI1O25WYxFwNV89e8gf5Utv2MgY9vnWLJFmZGOL6QGrG/QI8eh/jQx9kksIzD3RYOrkqxNjXHN29lz2eKAhmmvEwepgatvILlkEIpoiKeCrUoiNdD7wFBGHQMhJFNaWna5m/jO44KqKcPEBq7SxPIQCZxgDwOL0bZ1YvIWKbEc+GRxIxMaqUbDIkymAn0yQG/VLI5G7cP7cbRDoFaPCX6SPKJJl/Nms4YoXGRVQphkShy5j9NK1k38LSqcWv0STycs3WUQ65hb/EsMGeFPGxt+aVsuKfMvgbA/wiiMuwHr6EBlGpyOnh6Gs5JgXdm/WVzDVgZr0Nlf70QMyIlfSq8aJWP8oDpOJBpMV/d55SL7XQoFk2pFpGzy5jCGS3JjCJiZqb3JJ3kaSAsoFFoFoYv0BM8gpI8hg4kzn/FP/iFUUWzcuCswcWl2vI/PzG6JfO0UdtFBI871NsNHH9DMgM+3MSxehKok3/KWIv2C59voRPWEA7EWFgz7x4wU2nzw57vRaTRKbtZWEIXN6DDnAXnacp8YAhoicCnqdGWYTWgdrLwstv6+hCceH0zluVkIxekJ6DlnirS74ARcSkl1k5M48BzRtQ0CtSSWaFrEbfY4YGy1oxc80LzxZTAf+vmRdEAPVVkuSgm9cZh6TEKpj1ByQwQ2at2f4YkTjDbkDnX7xX3kLOwNECIdhv3p1dn5zg8EdIKrzY55c8KH3cwis06tcokgwZaEBdsDFA8dnQHjrM4r0es/4E02dWxA6QL/gXsFqBGsFilaCTqQwBtZz0D9X+f9YP4JnXZBh+Km0AVBRfQgJpnyTXIHgYR/mDTQgd1nxIa3sRsA0Im5sLA89uLDmsThsgRxTze7AEPjWMDgjd4SK5z4HfVQQOBwDZJ5FfXEI2zTMZLnp1EF0BBpxPbshhoL9Q/HgFjUGovDEg4ctitHgulA03JorBDcDG5caM1h18K/KeSv/GkphTpvv66iOSKXjn0WJZMJkLwKQKfJ8tkgKiNNDSjr9PRurhZOOIiazDqNa7NXlH7ADe5JzmgTUaS+6l8NVQW46bCs/4sdG8FTg5Dkqc5/CpvBQMrlMQq6r+7OpLnNyb1QP++wJs2mXh9+woT7DGA4HieA5SgFyA+0kc5IqhFHJssH7j5okwbGbsSWLeAIXhqsyXyvUp+jCKu1DSopnnZp2zCcng2yMsuChju8FHhOMtYLPQQdUD6Kk0f2Ia6WOxTGKyBUvCN8AQTNJqGt4a9beH13dxa1Baoy5iqCT8woXywfQxwVZM0EOA1opfwCAaFZmaEwcRChy1/EaskAkAUYTZgojP98vIXyo6a6WnZ01na4BQEWBNaFJCQtMwuuXJ6o6bPTv1INIV8qPXJsYGQ8e1OdTCXIIwBmWeDSAVGXy+W4NFL1bJ3DReOs5Lo6esEPdkmaw57kYN7agRwVPw4RlKRe0qeilaBC4Dap6BZFEX+0yL2/j6W8ufXw0iMxURNwYmzy4my8aMTacEemtw2ExkKJXD3gu3c45Jh5kZn+Zisxznkw3qUsMHv5qWRYMAHcaOpJAZUHfuoNAEBdWydhIlk2fgtt7KShVUFwYKQkiFYniFDs14OJi4kz3j637NeJ+wMvfmK45aCozAVpZxYLNr0FzqOEWgyteijGYqNa4LfIHE1hRT6DIoErju2rCfTwWFsSByD/ykWL1hoQeCDCQFPrlsm13ydQKJdFTIjQA1qDDrxQEB6TfNTjxCHS2+iJIeHausahr+R/Kgibpl81RKN61+DmJbNcy+3WeReoqupZKWXAVO1sUkvCkvQYycIsYJ8ZZzQJD1Dog5Uc7egw9v1SrcNN5haIHSAuLhGc2kAOalhbXghZ51F41KhyA0K+kbQ60CQZYp2jw5zb/+Mmaq7hWRTeUao+Gw9/0gI0EpwXzE/8TZc7AnDdj8wedpwczzK9K0vo7oqvpK79rWJuUkCQGrLlcBiyGKS3qE2Gz2AbN66n/5sp6xKosRklUFjS7V799BVPLK29FQ07lte6xt5SEPTmuft5OWnqoBshW9LySsBb60rKVSUGUkqWpaNy0EAiXy3Y6addSDtzhH9+fiZQKBSXRkbwxqSr49PztFnkWmG9z2tuddHF+/pSlQ6VBCK0Le5gkmYcci5yl9Vwt2cMHObGPU9Kj85NnEu9qLvjD6ng2vo9TUePKfBdLktB34uiEK1QwNaqcKSULFzZ50am3qK2C+De4XW0Vqu0DHavnrHuzrwoNlf1N2BFvjG0NmOcd4+eDMP93cDK3/mr28YdYA/v9i/d8MTVuEz1o8gLVbIC2JZCKcSCwqMMvRo0/5S9P+uQfAGzRpDTi5XHpLLB7IZOT4jTNT6LQkCsMrbN83IqtwAZAKke8QYidvoY5rdnJWKXPlItQOi9XKn8AWNxh4do+lSaI+9HHD++xJq0BJTTCLgHpBWU5fDvMH7pcFZgLqpdRyXbfglf7Yb8H2W/OqZ+W7t7HIfW5wB68azxKNlnqXqagCXfK84P2bniZPysznWig21II68tKm0LyuXEW+Wd8GUWZ2dqIWkYxR4tOimCeWVVURz8Fq9JxVrao7VWIBDEGSY2EKQK/xP5gervtwcXV9eXp85r4+Pzt7d73536cnJCIqNpsqv8QkUCBrxoSfmjFhi/iQF/5SlEoBRMvl1EEz1JA2nyQJFZ9dcd3zi9Jb6c5OIcv4nQV+JOdVbRaFj9/JZyXZpvED3xLfT9MuSr+7ptcOsIBMil2sk6q+yI/Zk8C7qex2i12lmUq8hUeiFuvaqrK1OojCUKyERFtkQ29Pj0/E0JamSatXuyJw1NaivIj72ctEt/6oYQY49LUMlM4uZAevkRNS5iiPYO0/oxTbdAIBNnUAW3cd84RdqYHoStn+ystzQ3SqBvSAtGb5ABtwg6928uTaDfsHhrUCW17UO6aukSpNZSKoqHzlBZLQZ/lUYpuX0Spwk8zNeJlzs5JuVkQhiDiXDhZTqVwJvcp1PCx1q6nggvSTXeUfoJqJscvTmoqicAm7jcfwhiowkJwowCZp8QhLdM9eO0SyGfCggh6Qq1jX8uJE9E3Tt890dy5Wtd2yXNYcntX+eVtFqmYIcmJj8ZibNeMQU71AtBd7PeHGqleVQjIO28KeNo39rBP7j96q5FuJfIVXraNNbDafwc5lm02EjwAPfPxCP7sBU4eMX762oRwwb/ljD5NtbCVyanzASF8+RnFrm+yo4Ou8UkYFs2GR2KkVyX6NpapLEZU8eMF57QNENF/iTGNdtIqfUShs67YMGIZa50WiD7FlvN86T1V/jQTuFfmsuEiruGf4K+7F7QLvJL+N0mvgM58WUNcZ/W24OEsceqPeBolv4xo5HV+uoAJ9oMFXyKx4M+uAhGe+MkkQ0jGqM1/Be/87Tnidg+Gw63hX/TW7T1876n02QGFAon1TDRHRbIYQ33Xwa7MrDoYC+7lplJwIgTZO53hQeBnW2Z2uSdOuhqoSpN73UI79o5p6lQWDKIMSSPBU0yn0CvKV2Psh4E3KKRsm7wGuHXl38zIMowfTwMsudrFOG7ObJ5qC2r4EsgUVJJeuPLD92q5UHa8yx+BG2eFMbCG6EPJGWyaJAT5a5U23/PJl0xtgix+iY5Vg0wmNxHBUYcex6At3548F9f+RUyAZXCR8uPhKd9QZDzkUZz53K0Bd95Mox4Zsj+dux4qa8Wn36oaH546dgxlCdIijkO4KVF67rwTRrkfAUT+9fClfUjZYiqMP+qKl9rJn4qaYiCVlbiossEqofK4UEGM9Stw0tPtaImQChs7rXt94i2qjnWoICrBDXxEB5oJoyfHTF3D5gg7sMLen9araYqZn6gS22Zn7++cyoEIhjpjGi3++WL8Irl+8fXH24sp+Ef5pdF4/kNITB7iCBhcceI4ugCjYWiTrH1H95/rptWr6Y3hDEXQwRezCa3W0rggZiFEST7s5GRoXSuh4x5G2Q5Q22R2dqlWXBKjiVTpz72UxxLcAI2cujo7dVQR5EBS7ZZaTC8joXoj03a48wNIS4Y51z/dQr+T5tMCJ88r4FvYqttnxapXc1xffRvaYCWAZX3sRHUOi4E4dCvki2uDliSQrGq5PWI6yIrV02kmnxehszl4n4GSTOPIhabIU0z2Q+GgyHGqXjhS0fyB5R9ub8N/ivPSIJfHqESONXHBk74YbyU4SitZnEofRosxgd/KmUMQqGzlde49sjqZSQEgtUyBKPzFo7QlKE2MF9WNQRu33gh32n9Oak+0dqlhoEQnUSHi4QXcccQF92QJ0C/dFXqHd3hela6CXbhqlHJsvdOINWPoY2D5jW02k3332ckvVpsgPTaTbDS4JA7LG6WTYlep0bLOE0HwjYrp6JXZcPq0SH8BTZqIdAOMEUKMtfYnC6q1qhlGkk4PfspPGjmk5IY6vRX0ZYq+EdxloD/tuXYChsBviO9FB2+ZbpsNbdX5dE4CtlmkKWuyxNMmjAj2QUlSCRARH4BZVfQfoSDPwnAm8Iy90Xd3avopE8++qMS78OuUXEhmrkFE9Ej/qCq20eV2CtwEjU1zadUtQuP0pk9qKAUAOVg1bFbYaUUINy1l1RdGYVQ0b2vF3E5udL+aZRS/kJLoi4kJwacLyh464b23U58rVXSw66teyAa0QlycZFVRdrtX6jmiGVwYhPreuZn1PCiIzoTrvAEb06L+dkkhRQwaEd7Na9bvyH4uoqEv4tlMxlN9xMR0sMTfESA4MgZz1gslQLWYhTLe6Z9IuU4xoEScZl+0BvDQNBnaHd5rmK8o95UUAyPioVX1+QTfBPx5fvjt+9cfpFfsP7djAhkIYS91msqgnSor5m2/gnWnbp8FPk9TU0oZi6yYm+H26Wohq9/riQ59dUgtzkSVlKqw9F+Yu6k60HmxoNI7nahLBW0XeyhVT8DCkdWfla4sETlqFbUTxKFxXY6Wguun/IQmiOyzsqTrZF5daFG6zt2H/QgaZF5JThRiNl6BAVDlWLS6EX/XGHPZottHGIaUCEgLrSfGAQTGx02RiZ3azI1Nl4XqpLtmZbZgoUGrArcsaen8M5NA+FewSnzrSagk9r+fS3T8Xb0FjWt5kyrAseYNCNNc1E6RWhOVhbmbJVM2SYcAirtr9BpiNPFiCIpyGNgWAO34UoS2dNepnisCtlLhmQF3UAVQ00Wg2DL4RaJ8HK+/w3GxnFYCpwbRFUNVRREdc7m8vEWHS6M5DWkJsXcBBS8CEzayJ/a5ewmtr6JAe+EsvkpdkZUNEtjy0nd7IbK3lFz7ScYM6sNf6/bIpotr+jX5Co58O+vbcsdMWtgvllQkb/S5GNDqMho1AUbxOMBT8W63mLgPCVhZaDXxup/VIBzZbnnZE9b1Ddw0SsHj5PBMp3w41TnbQbUD5b+FvX55omugX7Mw2LVWlJC9B9M+1GGbddyx+8zLZSmF44gHujdEPuVBqSwjMSRaBYQjx4J2kSNSFENCTDFIomublSyxG0LdX1y+U4Lp1AdQrVntDukW3Ny28zCl+7FVg0tPtoB5jWFVEvkxT6A4CXQyvcg/9dngXiO7YjIAwBela0Q77OJdSJbm20Rj/nT/OEy8L3qli6+i7QWr1mdHKS5/pveuGqnfiv3rdhn5yh986KLPFNSKzRR/eOC0ggAqjx1vEuH5KTp5g9jrphRAILl/PEpK5bCU0DiT0XK19IqG/e+ZIorMnpq2j1pfCDLXhS8XtptJOZaKkjkIJxeaCDTa8W3Xj+t9ueair+Nj93Upan++ZqdS4HXghQCi3aGlnns3DbcvCHzJ+ph9vaK1PMSxdRF+4EOUw+g0U8vJu4/aNVdUn4oVgSbxC6ebi3PT9n6cxzxaPRhMiXQrn9GuSq19fXZvHl28u+cJ88+7y+uSk1zNk6CNtMxzxqH7jYtXXs2F85AyHk8mkBV9VipYq3iSVWxWkIFjm+hYUAJYsALa6CgL+7HsaJ0fNX7lVOQFgkqssdYdbCq+lJdrVgnYW87XcRfXHUPnQxZ46oMBqBzCeLOb4g44efEXZMnztTNiv0StQxIdoXa4Z3W+vDANbLpVfckHJ4evaxF8TY6M8zLAc2Cqit3ygglb1PREKum76VbItPkz5dPXu1+vTSygctvB2Fh71AQwA5DLDodzmOSf0dawNGnvNAr3ptoTCNzxWc3rlTbeqy2m7thQZCCUYU5lm6D+CVhCnujvUarEt9ydo61ektK43gV48qVeQV5CTx5ICvJ1Yuf1jRSXlH3/4f78rmp8=",
 "summarize_e1.py": "eNq9OsuO40hyd31FmnMw2atiiS+J0q4W6NmqmVl4prtRXV4blgWCElMSuyhSzqTq4eoCfPDFZ/sTDPi+nzBnwx+xX+KIyORLVPX0LgwXuvnIjHdERkYk9c1fXR6luFyl+SXP79nhqdwVuTcwDOPjcb+PRSo5u3bYmmeZZBtR7Jlci/RQyktxzKN9XIr00T48sXgbp7ksWbnj7BAfuGBxntBbfASSQv41E/EDE1wes1Lag8H7Y3k4lrIauZTE7uny2okqoLW8Z2aRcyaKB4Yk9xxIJeyRJXEZS15axOQMhTJeZdzeJ/bghm+44Pmas/s4O3I5Y7fxyvaYeeMO2cF2AkUDB31mPsRZxsp0z2luYrFi02g0ZKWIc1R/xROW5oOkWMtL/ghTgJGX0SGLc2DKfv4vZ2Qj+ewoUe4LyQFhk+ZxhqpIwGWAdnkoZBntinWU8we2KQQbaw3BPuCAwSDdHwpRslhsD7GQvHoHu1SPn2SRV8/gjF31LMu4TGWZrmU9clwdRLHmUg7IjQeAztIV09MfEHnw4ffv2JwZm7G7max8bxI4Lh+5o00wWfurdTLywtBf+xPX50kcrBxjcPX29u3H69uPgLUwNiLliTFkxg/FUfLwR3yMV3EGLsTHVXpHd57x+7iEkMCXn3gpilsRbzbpmuDTjIsih7kBa/0Z3yHtfZz/yGMEq16/F91XucXX3RNY/S0+XXEYTb7Dx3f/cJ1zsX0yloOPP7y/uf1zRbZrcWs5bcW7lqmWhuRoFPhFgb5hplo0eYyxty7yTbplWbziGYRRJ0yAL4NQeQcSWoOfrm9/eH9F1idupvH+n2Pz43ZVWsiBXr7/9tb87vc3t1dXVmfQMqyhRvooDg3Sx5sPPSTiVkNvVyYQUcAIiBTbHKrBNgfAeSs2zfTbm+9u+LaPBuMnaCBcM43CfUGusgbUQKcwbx+3K4R5+/fffytOJ0FkkKmyUUOhmmioiI0G7KrRjLRUeP837yis6KENqWbaTtBEW0qeiojDF1dXFGUEcQrww+0NhfNtM4Hh9YGiS6U+zHrM3PM4ZzcuK+5hImCYoqSFoZcd9zmlqHptFyKBWB/cvv3Wg0h7Jk6tQJuxxcgOp0M2sscjvPpjvE49vE5oxPXpmcbD8Px1TPDTyVKp0gpKYjCe4HRAJHxiFgZElNC8ccNAERoTy7EaIfiA4KdhzaCK45YCAYnihy0FHFKARiZBT/RJc1W8uvQp5on+xG1kCIhySJQDNU7P47DRSPFVkiuNgkmfPi2Otn2CRv7JtKHsOS1reI2rtMNazms5ANcT0VYOdYjqSOlNsA7gXcAQMfXChpTvNleXBHNpfBpUxGkhKsOPiAjaBG4O4YR+RfGiehk7LduMWtZS6DhxMbEn3K3to1etYqKsrMgqn7mNJxxSYOw2XlG2U1cFqWPOqRVQOUDZPmioa9srToTjjhvPKC8p+HFLL+Wl6bi2PeWGtuhhcw1bq0w5Vi0LJbq+Ekw4bpRsEVeZhqirORWEvlpQbmMk12kCL2iZQV9buA31KkUpy7REdcbNGlIJYdSabce4XoWT1sr2K/KY4FpGV9OO31jBpfGRcobfGFqRUzHpBU1Mjl0g/dLJkT5VfzpHtmpCyWFn/oo06b+SJoPQBi3diQ3sfRukcnwbLOyC+YbM82yQ33Xx6kxce4I3DxHwBmI7ts/hzYfiEuam9oh7Z3NlSIS8KUJNbDCxO0Ve/hiJTKZIMQgIRJWpXuAR/3FA0tkhcnGQPL653D+XMD0ILoByEdPBS4CaTJHdlMYcfAwdJDlRokyRZwDao+IBIYFeLSVa+dJTlvJJTLKBQ2gOmg7shTYK8NkZhWpCzY+IVTgK8E1rNLanXSZV0vQVDgnkkToOYrtkbZfIu2Ni7jqonaNZgtag28RzCRFsGaKhvI6hKHMG5F5n2mgxUv4k55JXgAGCeBMyGvgIxfC1vx10gTceI2DYNlWVPZ2QYgHVRk08Mv8E2UHI4MiUYg3kmKAEE6KnPKtvYFDuo/em6s3jQT9/gpDjyq/45KvQctG2xB6ZKI5jiogxPXshmU3bsCN/nTw9ihOIehfJBpWLa0eH5IkRjvhIRmk3VbE0paEJsXdQu176DMY455EfyLYueQCuQNKnYeVGtCSqN6YQcECNUMusghXXSSeO6hwKGoRke4pnkpTix9G+R0xiGpIvpi6Oh2Q4PyQJJiHi+uju80mUUhblOUpb3QvZNqwvE1SUJiaUX4KTxEn2Jos1+Pri6wvlnuYCqpF9HZW6KFsOBgnf1J2JeTzIUvB4P6w68yEVlENqV6wZCVCKp1ndEpX8sYQk2TSl9nrH13dRQacCZqf3WxjblAr7i9/BFRjV7CARG3JXPMDwxniG5vVldtpXXz5riV6iZxTp5fIZZXrBwwVj2W0yZQkJXMxbQl1d/+Hd3/7445Dknd+KI/RcCMkf1/zQ7qnt38EuwZMP6u1aiEI0ygpeHkVO5TiN0RnAnGXQoJsghn2VrssbHgNzE/nY8pClZZbmXJqWpRhqEpusiEsT8RcXznJhxMmnoyx5AvsRxw465XmJZxYJL7nYp3lcpkVuLC2WbhRXnkmuBFEeFHF+J011NDJku3S74yJa8RLQtdtoV0NXFQIY1aB3/GmexftVErO7+xn8X0BlBGJCNyH5vEtIa4CnNEOW4t74ApFGow877GdT9huW8dwkXlZjuE8Am9ZvCvYT+xVzOvB0gEOPC5pcgihsPtdDKb7OOn4GqDlz6iE8ebnDnRyMseVmOlQ8rC6Skn+hiN4tF6MlMGFmCqCfLHbJXMSpMVDLT/WI9p4ioS2/2ZfKmLBmUghwOXc1R/AVTbBUkqt6gWT86V/+3ahA45X2n8V+C1pxvwcOS4MAZpBgXozB2ZlnJcPLBgCUfPs4zc1YbO/n1EYqqnQMhdFQHUnZb8X2iIdfH2jGTLg6HoSwm0dRAmswslqYdpwkUaxRTOPiolrJsILLpwOf4zEUhtE/HVPBk9aKewVfn/110UH8GEbpzTQqEL2SABsXn6ZHN6QoSVdrQDDqtBPjdFAFCB6XYYwgmLQ1TXC7IeIHA+/XjmHZ26xYmcabyzeXmGvmby4FVIwisfGYzrDansFhjBGYsGFRJ9JEDkA4TiJMAqYWV3tZ09ly0BrP9o6gD8a4sS72hwxWe2J0w5VUsCHvaWOYpiKxMNQRYwQL2KAVq0Z1loRcMWSLpWXHhwPPE42kzYIZZMiwKI6Eqx+oOp4DypBWdcdiUE5H6kBLn2TR4RUYUR9bzTpLMM0T/lhvIHRGmoOfRVxys6qxe2sSpZNNdlJao5VMzbI6KSatOmlLzJhYGOgnY4knaiOrQxuMnhdlxaLLlwxc5GWaH3lXIBdtAWSFC0RRK0GJRRFZdmArywH0vojJkJFuMH4R95ij3BwjSM9DPqqV0Xmjg5HwDAxBnjpVE73S148EUGHaSNEHU26oztjnTUlA6+RMXVAFnJJVVwhnyVLYV5RBJ3RHNx+e/iklq9jFzFhxI39cNPSsLktq9OatU3N7g0OmcLtw6pB2zvAobNEE+HJBwbvsWxxgifaFQu0AwFaZwM6PB+77uFzvjCqdE6bFfjPHPtZTe7YBmapM4+wskDPSQPtUKlKnPm5kpaxB/QPtm9Vyw9H6QLpv4Zas+WVs9IwHCWFhtld8te6W2gJ9DAz413F6rkDwrjMwHVW+fu5JrPMclLxtFu3sN6vSUp38ZnWY9skJWHOIAmWHXgxYfUI4RvghTBwzDrPVujyH70aoBwDhbUgDUHBGMsYEDsMtjbEQvcfgQ9cRR3j8LZQ95GXsXc7SV4qW8cqrtEbdME6ie1mN0zuM42uk3QrD+ukM4X38GEHARTWhurxGbrSMQaf4UUWkJJl1tqmrzTNkayp08hzhPhYLooUKa1pn8ChPUgDJrtFaYTLUYLVBfALGk5kzq/Y1JgKrZ1IYUV5lBpv/1xN+3D8CoXbOxhFIg8b6cIz2Beh9Mt+ML19xDmwZ66PAtPakPQFbSnuwv5ucM+yBx3eRkDLapqsIqGhaSKyeWj2VXFbbZZ8qlsFv3ngn4fmiywfo7VTl2Kmf9FdbowKx93dJKkwMhhyKYqwAh9Bvgd2j4q5VEOqOpklVDylUaCYyoWqs/RUZKrMC0oRpYKcITSH2VnMDnsE8RZLm27lxLDcXIdRUELU7yIunO+KDSEvK/FW/9nc0YCpY3MV4lqDv5ZzaOmrRRsuTbUZRsem2Uw3f6wBIguho833DrlRykqzIsRla7xi2W0/VZ/FyF5cgvIS8/qR7DRZnsqCxcgc7qM5ukKckfpAG98X320v4fwNtoD1QqZk+QGOhsCcI2Lsj+Ncq22gHyp9ME7d1Cyf0DkDwSftM1FJbYoIf8YFi0odAYhVP3I9gRb1CeK8GCFTTpR6ZvuN+gz9SqKLKrNvim5//qOpU9ZmLUs2v9S6uPoYBTRVqOy65pO9yJ5+eoUuqDV9V28p2UL89Y7KqhHqpxJupcVQbBqEfdzybdTdNZoDZlcbK/OgTcGz2BK4q4FH7Cgpu5FUchax/UaF/X5Gld5y+QBvKGiqiyBw/qb1vCR3ogj52Lyp725SazMRa9n1B4CgXfeBWYhltS/8KaX9mBsAZDO72pwKaRMXXUoOI+pkALi4u4OENpXMNomjhmUNECkGHtZ9BRdrz70sDWJVcX4SsVUk69SF24T3FO00HRJrQrf+dNBcYeXU5Q1G4tHosIVtTGrJhke6ledqTVNot9su6i/pFplRQ7mHj+Mv4kZH6DE8pNZjUKEX6FK7VElAjNlS2/MX2S/tQG0y1XWQ0q9+KEq9KPDrAwJVPFHS7oqqFjfGMJyM4AUvHpJeWcV4so9MXJ71+ADEjXFtnasgT13bD5kyl/xqdSqAvLao+SUWspSCSr5WsGaKOPQ8rxH5/0go2a2Y7G6R2HqoKEQXWYoFmn50RVLNtDpvqPFC7sZcKFmDYN42/h0hqSJSWVX7Qu1kroWDKoPxdf2ljpvxi2va/Jm1Dl7OSRXaEbK3baihLUW7cPnexSB6AAOXUVq30a6YLUUbVn1QZ2cRfdrE/BJZ9nlHfDn9pBq6zaI9FJ6eamFTrVUmnlcv/7xWP3vo/X/ND0KS17H217Gm0vfL/vDA8jT3wb1o+0dlLq4alH+Zt8EzoKzqeZftAYjmou2wk3CpJCyHR3Lom/yq6tUCKWKMzrQL6ZrKnmvyV/ukLFLprbmN8qH6aqA1yL8/+WJOZpx9Vhs0vFa3ZaWHTqpvoi4t8Ydg7VwsrGaI52Of/+Y+b//7Xn//4Gbd2stPMdvmLresN3Y6eHMLWjmpMXAHiMVqnpaXj+WqW4rQHMcQmpjqjb4yzMVRpiAHUCGKCxBj3n9mf/u0/1cEMHbUM9bszYvqIBkzyXKHVCrUak+rHqdCVUKGvznuNf8x19JIkFLIwdKZHUTEs0ryPBAE+gDiMqA2NIjrViSI8yo8ifaojYvxJ78cnqI/3149padJBP2D+L9M2Gxg=",
 "run_matrix.py": "eNqdW+ty4zay/q+nwDKVCjmRaMtzqV1tlCpnxjOZxB772M5sanVULEqCJMa8LS++RKX/57zmeZLzdQMgQVl2MknFHhIEGo3uRvfXDfirvx3UZXEwi9IDmd6K/KFaZ+nLnuM4l3Uqrj78cCnkfS6LKJFpJeYyjkvhngz74uQIP6/w89oTd1G1FtVairDG8KL8BmPCeSXOzo/FPEuSMF2Ufq93nkrx0+czAXLCban2xSKswlLiIZZhkcqiL0opFx5P54trENYfBCgJeRvGdVhlhSirIkpXJbdi+h4Rz/IqytKS5pXiVhazsIoSsSyypMvhT1fnn9ApXUYrEVb8LY/SVC6Y46jyxXkaP3B7Et5HSZ2ItQzznjv4Nbn3RFSKQuZxOJeLPs8fVaptnhULEKE1KvbfsswKSDNKRR4WYRzLWNS5qDIxGPyWzUpfvI/SqFzT3NS5FxZSlDdRnqMlS0GURt+tZcrsaInSdHU6X4fpSi4g3V/KcCWFexql9f3Bv65OvVFPGH2Kcl5EeVUegFCQhBDbvZ8/YPo6hwxlmIjvtGDEfC3nN1ldfU/Mhbeh+I5+D4fqvRDfJVkolpAYXr4X/4056L/BAPoUJ0M8aGVCK7MwzqDyWXQj9UrFKzEZDBKJqRal8H1/Su+k7FIMxRG/3Sf3YiGXYR1X45er/qd/n0Dxq4fx8HA1Jbvs9aIkz4pKhMUK4iyleYcy53VRwKL8ZV3VhQQLpdCPTafy1jyCUQnbaMavw3IdRzPz+luZpeY5a8aji7xvXupZXmRzWbafH5rHag3BLmCfTQPNxYaYhxXNJPSHC7z2ehjqL7K0Cu6KqJLB7KGCLS2kGIvrom6YlMMgkUlWPARLGZbRLIqj6oHWKYdCfCXS7D/hCPvy8EjNdBOuVrEMsLWjIktptwVlkkEfmtwvF1fXlyfHZ8Hb87Ozj9c7JHpvzz+9//jhCjxsWNHOydAZCQe6PsBPoLaPT5Jy+rrD0eMOQYZ9eI3Vd3u+2tMTDuc6OP/3cUAbBXYii6DE1p+vg9PL4P1Fl8DrLyFghm57l0dYjxMufqvLire7XC6jeUT+LVvC9DAmidKQ3IjT+9fx6Sl1104HbaxH4d5hH3vCLaEkeDfP6b09Pb66+vj+48k76j+Pw7KMlhEmiNKyClNYidO7uPz46To4PX/7M/o0BuKfZvMb1+v1erB7yP/O5b0rsPFhu2ljqH7zQH2aF/r1OzaaX1dzz4/KbJkV2OMutZe5nI8dzaRjpoizlYtv5DD0TOzBW+5GelvDgcDDVu7SmWz2c+GNvv5x9PXZ6Our7VRsNNGt0xfLuC7XYzLddtZwodXkGr/Tp+hiVhveQSpy6K+iis2/tLppS5yg+9SzhUNq9Yl06YKA1zfb2C/X4dHrN9zor+X9IlrJsmqkzJ4jgK9xSUSagyTMEQNWZO5bboAkqWdFrpv6+WWO/eY6fcdrRXQjH/qCzIP2KvU2vcaO13TSpCfo7FPYyl1viu48zDRw52gpHO3+HNgCT60Ht1MWYVRK8ZkGnxRFVriO8pspe9Kw8Z/fldHv8nsB0y4enI7YNEktDYpsWuQ0Qi+uTiO4cQjDucFOO3rxYnjYF06ino/oeaWeXx5ubdpkMURmMhoMp554oQhNuAkt0NadLGj9WOtuIy1XzStjLNGQMmqb1VFMRsQR0FXGtAMbgmUcrtRjX1Dsot/4BvmQtcG2sjTIsmT8PsQMeqnEiRnL4ZzeKMKSBj5hc7WyN1BkLJbORr9sxWDTjN8Kft46POS32wRdFacTh/gJNERxptpQlGJIB+g5idjqor5GMiQQmdaJLLDtXFDziFn1DYYDcytp88ICgEscb2qWE8vUJZKe+NtYDJ8znRMgsTl5QgZtwDwUtImaxUEHOnUWoZjPQwAe2k5YAXic0MyTw+nUiKBpYLER8Q06axGBW0svLatfifOclo15wliwajJgshFwX5YzSwT4NHhbRkVZifO6Ol+ecYDk1bHzhZclzx7LqqLdDVRqTTEPq/ma0Btp/UZKIlzAhVBPorsKASFXhLqA2+aEOkk966wuAGkNGazQx0yyqFy9UPGtAEjGSn8dfXuCxZ2nu6xpyQEs3ZBQvtkYG2nwrTOFXQFuHxfL5fsollcKqg2WQnlFP8QHTwxi4TaGiNcGIZOFAO98Y+/NidoRzoADFAER+GqQc7FH4DxfYCV95mnab5WqN98SUTEOCnh9UA1obLt7aJ+YVsSgYAl+XctJ6ulpK3EbZq+w7OadA1BDIMthvam8i6NUjh1wKFNgIahk7NTVcvB3xyPIA+i7iK2dSYoBd2SvoOS/i+bVJXQPt6J6WuxYLGCELSBqxXpZPOIdsOzbEJBYjQT6PMMuLKIwpk0k8cUkDceX79/DOueS7aiQAGkzOAlyZCryqHZwN89qIFQGskSTBBsEwP9VEABNxMu+aKNekWWVxTV99tuv/Epd9GMMGKEfgSATct1dUv1HmKNPka7hYyUrzYIG8dbcrKBmmq4koX89gANlw0C3lyXktstED5x2ui6zeMEutlmiODBT7OnoJzeLqHARexHpSgYd7OpL+JQbjUHsUcCFIe05NcsBobtoARPTALXTl+yx09fae86uFIhwa/2s8qc2hPnPTE2Os4UyTIhUFVTyHhGiyz+Fxj2Uhr7izUTyrrU0SbZaivccJyCluwem1eX9/mgQM6oyFuaU17Cok7x0zcgd5vfoHjMy+X7DQ3fIM1ajPBOltZQ6u/SrT5lh2RcGHejMgHChkgIDggD4UKsDmNISj8pOO1iigy8ommOWnloL+sThDAk9MH8KZ+YYKEHggbybwjHkYbmm0euYNzEKNZfAaiXZFsCqg38pl27sHU+KJTwsHSIy3rTz6hCq6g7BI2NV7ZZZqxDN8+LRpx3frNy0ThoAqqG2xlv9Tpj/UijG1FX00Y8alD2yqSf+41EWUFC8NXlEa720IHtlquNXXEiiEoZ2yHa5ZlZXHMLISORiJM7CW5mqJZaEHQgJlMgnjbsvEf/J1xeShmn6tO0GhcyLbFHPo1ks+7peJYsH8TNn4pAF0kKu6WjilWLKt3jMLNRzfn6mkA+4jeUSKSo4rbIMUs0EWU4CbmD+pcIuCaEZgi5QQxzBBtQKNVQhCVVULwDM5LTGzm+astJSNQE2PgtfprZp8O4C2Z3ky/IGEzN5q6qJo7o5084mnXo+h3u4y27mpp2sZev7HCtsv+tL7f57XSrh6XjBFgPvHlY1UK0YUxKfJTlQo1w4rMimk71o1dVuGe3zXRjbV4WcLw9bpKWUjA9acp0uuAM24qQCa13Rsx2dLKG4jUsgQp5fp4BWN24SwRrTVXe6rPTLh4S/m821P/qp7lQ2Ro54MxK3zCkgyC1xCjK67KR8tsuJy43JaRFifjr+fBycX1x/PP8UfD6+/Hj8w+nJ1dbyZ03dicEXvCLSTXbWjt7ZeG+8l6O8ZIAMG83Gi7PDxLvyQUZzaDA+raVvdEigk0ZQpc//LYtSV3/xWgJK9aOO3vkrWTHa7XJFZwq2cm32o25ockysDlQFGt93ynOs66Idrl+sCeBP0W5nWBjTeG18ap5bIbb09mzMjvg5TQzaepY9Ys/nqZKIKsrOSSvK8Wez0qK7zkpaKsylJtt0PT/F3qdHGp7XQYLX2AxuGtjW4acwY0jjuRilyCoToiLXUvURYgP2kJfrFW+pUkX2sRWP4ilyKWV8k28gpW+mW88xgYZno8IdFb+SDIacpdFcOybGx+02azelp7IZ526mshY4cLu8xuVjQrpNLdm/4AFN0J3fLcaKLmVqC4wf44efZVGMrYFX1+/Of7mmbOl2jJ/Ww1XFQ9crBTScPB2yDT43GJMC7sKoeuUaanmEyQ+7SMx8U15N16f1SEUwqLKAgjQ7cNW0Q0KGlPDytH4BbYb3BSTwQgwPj141PeX9XOaVeLtGkLxQk3LYGe1lh6Z3907TJJgyDvNyr/rEwOi213WVXWf3rM+kxG5sJch2atFx17pcovhmdhoOKbRR9rqv6PRIgTYFrtAi1lUuxk/aOvTUinFanu7PUoXvvlUC8p4m3TAHvKXyAlX8yRLXTiPAuktMUy9OevaYyVgc2t9ahNyZnHuQ6iwxtI3fg4j/D9iKVbDELvU8u2jEp45hqis7u7ilr0raQOt1HKu4TuYK5mgHE7qibIYgFOXub68+W5RdeC5Z3IJ1CnTmZEpcrWbuVZF7VIoix/tm5fnimHkcUPFyIS6vrpSDSDPa/3nN6BM8At61VSQl4wYkEBD7U3JE0/N+mPtr/VjVBN6cooNz9FYhRLS/K5pNJzsHbTssQ+AxTegPfWIx21/JUe8+3DL2W3jvHvafIEPzuvRPQILGTn7z+vXLN5bdQ0qrQnlYGO/F5fmHy5Orq+Di5PLtyadrHzt2ESJv1BMSPDTIUKdOdU7nHa6FQ6hCFBi6AcLpHPCNoi5vQfNhwrVvpUPFASeCZM9W8Fvqg187hlGoRZJD2qaA/cgEuEMlC/I1dAoVmOOdkXFy1gS0bQL4V7VZiB4aCBMY6+mE86ZVRVdCwiMTKFqaxVGzWnI4l0dqoeS8uHSubaELoPetPsnC3RW0ZOnk7S8Q3nq7CfET5QnVow/vSYnJeKjVrmDDRs01+ke5/QLocHk0bsBDcQTsINoK0dKhlbbfd9dOvaGn8YYdnSoW1kis+fVAnbL0xZG3/RD90D3KMQvh84weHdpengwPKcdzlkUEEUGZP2Z1Kf9+So/6WJ4e6WSe/pUxolSVFSW9nMmqyK6LkA5FuT/2c0GVfjtRd94TZeCTUxlSJ/P6oei+livH1GiQw0VITyEtvWLXYPRusdFxnAsCqeI6nPmvAOThJklQ6tyVHC+FBLoR4TYnEidDZL6U+Ervn+LNoShNSboKZ3VMk3LBlfVAx+JIm5OwgMcI5NAchb87vj6+Orm+6ouzk+sfz9/h4fr4B4VICI3ySRgrWqNonT3nhMu4vS8Cymz08G2vLQpS9QZfFBnSrVUuNfM+qpXT7BMeMlEkptOJ6eyT1d43BY6pbQ9vDv1DLXPAeGwCvlTCpTHAvbROAm7GcxlnVdmK/TRLKdMeaC+AGDggmQ/U0QqdZ6/4CgEVRNRNoCqz6Il7RVDcZcWNLMpG5JyGkz2CsSmCt2uNeaGZUBBbtaHnZMrSZXkWdMPFGqNXS9+JCy45ZwTfzBKRAY7jMJktQjEfCXfw2PLmkyOqOUzgotU/9Fs1vZ4KED60IYVaD9hKkAsqbuh0jZfleXo+evGDYCUrynaDoA1B/GmiiEzFt+M9O4EYV/PTw3DaDlZL1qPFwYES19QPcwRQtWJPn13fyDIPUxYfxUw1bSPhkVvSmZRn3uFaSYLlMxI2ZVdtLc0M2riSEOJAUnY7Ju+r5cXn6rq0yc/+cbGqyWou+Iu7kOouEgDxOAgW2TwIqL5ANxbosgafu4ybwZfh3bt2wI8yzt+brp41nx8uEEL1RHQYblJpOKPqIZfjC64uF/I/dVTAa7fA/YnxBDGcLxtQ/NW5dP23O9yc4NMbcJLu4j1LiLzBYE7HVU/T8tcZ5dic3axmBVWbuQ7gPEuZSi8wznUWwTGM9W7TNQ6vnYCuB/0hh3Q3DMRSSufHzrfPD9BXxf50f6Q5cb2Q+8a1bE6mz9Lg62jdkSxMZA3PmwCyA6tvO9/w2WFUl9Gjmksp7dj2noopyo9frf7ADgg+DrJ0QFi9L8I57zXguawACoI1Ok+X3NfYYWPneLEQz5R+qfSschcBH6RK2Hy/MUnkIoJjix+e11Ieh6llT0hahgM4tCJDamXXnPZzZ3UeCQIGw5d0FzMzQRWu6vh+NeO7AnR98i3WPQAgcn8cHA5dquLxw9LznueS/V5HoWr+4WAWUtrF3xk+3lMghASepwf/qmj+FSuJo7J6QpmarQtOa/mgghkjIbfprUluwWWKwP4HK7+J8gEH06c2EB+TwUcWY0dDkpGCPX9oW+8KdXcDoN3dOXPzNKoIl3DvahVg9Z8iQ/fCgAM6cmmuveplEIfq7hWtRm0ZauPo1FQouH6olbZTXKCXofhubPcxb21YtMp2aiZp7l6pEUkNmDST+L+6kzIFRaLdUbzX06cmj44mwb99QY6nbg9v9dlXrtd7C1AezmIqvk1UMWKhjnDUDZLGzU4NDl50rgpd/XLheFOqaODpBXco1+q6sftfg5e65tdc5NWCaN4xnduyYCTL9sbJ2GM2PH2f7AYZbsrH+s3hXMn1N7w/NQr09cAn5L903hnO9NnCxpTfjdim25HY6LCliXlbbTz6xJJxE1+UY3mqJ0umppuSKX+maz56Rj0RB59AB59px/B0Y7sGXUfU0rD7kEQ2zQwdbgwXW/vwSlPaqeJ1RXSm71xrftWqWqFoEo1Q/oRgms+2OIwoOiJQ97vN5uIXwu5asACfiwxbjKPuVO8P9gOYt4X6VsZkLKR780edGDw+tbd5fXRtpbVcqirYYUj5BTp1oy/Hv374oXA6qZuWpMq2H9/GwPKqKK0l7TIdb0Z741Jn5FfiPJUDCqZFjb3lmkg1asVtPHmorhnw4TAN4MtHlJZqBv2dizKFvpaQClfpoHPnkTfuhBzidM8lFdaHyTlcs6usSxNPCV//JUfXB1vp3/dwkuCMmynEjXYSn34nr9mTzu4Q7LdHTNbpR1aFMR+xJO6zuaDKKXl/EHVCyS/fHB5amRzXhij5Ux22QrPxf//7PxueZuQPl1vx9uKXwZrKwapQUOZ8+rjZYXYrmsR507C9VemZtQ/5OiihjD40RwMgVAjE694M/T3K3UcyA9YZ7qjTlDL0xt/Qwtt1M4Vt9yRFrVprRGyYFfKnJAZu80YvF40kIGC/YQD+hX4rMY5e+0eQzbrfRpYNc2M7HWMjXWOwstFWG8ZJ6BTZCt0DMZzuRHwVmPiva9i6gHConlPVeSxdytjNpe2RQzJTpkDtjUujIcr2lPNdAMjkfEgyme8YDp+HtGUF1hQNbyKCHjvasawr+psfigksW93JGBlvclrGHdCQbF33pjNTw4geve249Mav/iHH2ru1XCsXQHO63frK8+UVj3xarApL6o5uezKgzgAOqNx0MM/rKF1m6jiAnlrZdM99wX1KlVyXjum7SsN85io9L4960CKIHF+J5hEWDlIEyQYdquNwGUPdWFYH648vtJibS+oiiTqA2nNLhQwcg5vro3vhHHUKOF3vVJ73eBfbOzR/QfbT57Oyr26NJ/jZmNtbRt98z7Rs/5aBRa7/Gsq/5nugF1kWn9zLeY1sgs5ZAl25G7dOlNSRo1urDv47LrpAhFa/rGcJFGDu36kC1hdcwOtU8Xi1047fw2T0xbAdlkFT+XeZv10foUrh5o/nxkRAX7DbOR+GEdNxle6pru3pUewrdJ3eHIN076cq2dIfc0z5cj+98dUg/luQQ48qbZZOF3SWKzaq37ZbvT9UcZjwL331KO/YWOcb/ZbHreJs2Ov1Iro0TIYbBIxPgoCqcUHgmD+lofvJVw8lHBjl8C7X6hCG/x8hx3q9"
}
EXPECTED_SHA256 = {
 "kaggle_environment_smoke.py": "ae1c29176e8cbd0de0324841d29fae757240ee958181de209c92fc79e05fe2de",
 "e1_memory_feasibility.py": "e09eb6b5f18fb89142b9b258c7fdeab4d74818b68ef4856879cd58215740e69a",
 "sgbr_c01_e1_integrated.py": "55c60b728d8e7141217e39474675b4c4a317aa7d79b1bba376b5a8b9d254552b",
 "summarize_e1.py": "1fb8686f26924aa2c8c04c83c1e6c671483daa3a366356d2486e9c0bb9ca054c",
 "run_matrix.py": "60cbdc752ac3d35726498e2332f562fbf6717626af76bd60a9a0665f95500da5"
}
for name, blob in EMBEDDED.items():
    data = zlib.decompress(base64.b64decode(blob))
    if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256[name]:
        raise RuntimeError(f"Embedded script {name} is corrupted")
    (SCRIPTS / name).write_bytes(data)
    print(f"{name:32s} sha256 {EXPECTED_SHA256[name]}")
sys.dont_write_bytecode = True
sys.path.insert(0, str(SCRIPTS))


In [ ]:
import datetime, json, shutil, zipfile

STARTED = datetime.datetime.now(datetime.timezone.utc)
RESULTS = WORK / "results"
ARTIFACTS = WORK / "artifacts"
CACHE = WORK / ".sgbr-cache"
for directory in (RESULTS, ARTIFACTS, CACHE):
    directory.mkdir(parents=True, exist_ok=True)
if RESUME_ZIP:
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        archive.extractall(RESULTS)
    print("Restored earlier results from", RESUME_ZIP)

import sgbr_c01_e1_integrated as pipeline

pipeline.stage("Provisioning JDK 11 and Maven (checksum-verified)")
JAVA, MVN, TOOLS = pipeline.provision(CACHE)
pipeline.stage("Fetching the authors' pinned MOA source (f62f7b4)")
UPSTREAM, PIN = pipeline.upstream_checkout(CACHE)
pipeline.stage("Building the MOA jar (a few minutes)")
JAR, BUILD = pipeline.build_or_reuse(UPSTREAM, ARTIFACTS, MVN, TOOLS, PIN)
print(f"Jar SHA-256: {BUILD['sha256']}")
(RESULTS / "manifest").mkdir(parents=True, exist_ok=True)
(RESULTS / "manifest" / f"{TAG}_toolchain.json").write_text(json.dumps(
    {"tag": TAG, "run_args": RUN_ARGS, "started_at": STARTED.isoformat(),
     "build": BUILD, "tools": TOOLS, "script_sha256": EXPECTED_SHA256}, indent=1, default=str))


In [ ]:
import os, signal, subprocess, sys, threading

elapsed_h = (datetime.datetime.now(datetime.timezone.utc) - STARTED).total_seconds() / 3600
budget_s = int((SESSION_BUDGET_HOURS - elapsed_h) * 3600)
arguments = [
    sys.executable, str(SCRIPTS / "run_matrix.py"), "--upstream", str(UPSTREAM), "--java", str(JAVA),
    "--jar", str(JAR), "--results", str(RESULTS), "--data-cache", str(WORK / "data"), *RUN_ARGS,
]
pipeline.stage(f"Running {TAG}; budget {budget_s / 3600:.1f} h")
process = subprocess.Popen(arguments, cwd=SCRIPTS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)


def stop_at_budget():
    print("Session budget reached: stopping. Unfinished runs restart on resume.", flush=True)
    os.killpg(process.pid, signal.SIGTERM)


timer = threading.Timer(max(budget_s, 60), stop_at_budget)
timer.start()
try:
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    timer.cancel()
    process.wait()
print("run_matrix exit code:", process.returncode)


In [ ]:
import re, subprocess, sys
# Runs stopped at the budget leave a link to the stream in their folder: drop it so the zip stays small,
# and record how far each unfinished run got.
for link in (RESULTS / "raw").rglob("stream.arff"):
    folder = link.parent
    link.unlink()
    log = folder / "java.log"
    if log.is_file() and not (folder / "record.json").is_file():
        tail = log.read_bytes()[-65536:]
        found = re.findall(rb"(\d{1,3}(?:\.\d+)?)%", tail)
        (folder / "unfinished.json").write_text(json.dumps(
            {"status": "session_limit", "last_progress_percent": float(found[-1]) if found else None}))
        print(f"Unfinished: {folder.relative_to(RESULTS)} at {found[-1].decode() if found else '?'}%")
subprocess.run([sys.executable, str(SCRIPTS / "summarize_e1.py"), "--upstream", str(UPSTREAM),
                "--results", str(RESULTS)], cwd=SCRIPTS, check=False)
archive = shutil.make_archive(str(WORK / TAG), "zip", RESULTS)
print(f"Download {archive} from the Output tab and send it back.")
